In [ ]:
from pathlib import Path
from rdflib import Graph
from datetime import datetime
print('Imports loaded.')

In [ ]:
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# Stage 3.5 corrected TTLs (input)
STAGE35_DIR = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage3_5_corrected'

# Output: 4 per-jurisdiction TTLs
OUTPUT_DIR = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'per_jurisdiction_merged'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JURISDICTIONS = ['gro-uk', 'gro-us', 'gro-ca', 'gro-au']
JUR_PREFIX_MAP = {
    'UK-':  'gro-uk',
    'USA-': 'gro-us',
    'CAN-': 'gro-ca',
    'AUS-': 'gro-au',
}

assert STAGE35_DIR.exists(), f'Stage 3.5 dir not found: {STAGE35_DIR}'

# Count TTL files per jurisdiction
ttl_files = list(STAGE35_DIR.glob('*.ttl'))
print(f'Total Stage 3.5 TTL files: {len(ttl_files)}')

from collections import Counter
file_dist = Counter()
for f in ttl_files:
    for prefix, jur in JUR_PREFIX_MAP.items():
        if f.name.startswith(prefix):
            file_dist[jur] += 1
            break

for jur, count in file_dist.most_common():
    print(f'  {jur}: {count} files')

In [ ]:
from collections import Counter

# Extract first 3-4 chars before "-"
prefixes = Counter()
for f in ttl_files:
    name = f.name
    if '-' in name:
        prefix = name.split('-')[0] + '-'
        prefixes[prefix] += 1

print('Filename prefixes detected:')
for prefix, count in prefixes.most_common():
    print(f'  {prefix}: {count} files')

In [ ]:
per_jur_graphs = {}
per_jur_stats = {}

for jur in JURISDICTIONS:
    print(f'\nMerging {jur}...')
    g = Graph()
    
    prefix = next(p for p, j in JUR_PREFIX_MAP.items() if j == jur)
    jur_files = [f for f in ttl_files if f.name.startswith(prefix)]
    
    parsed_count = 0
    failed_count = 0
    
    for ttl_file in jur_files:
        try:
            g.parse(str(ttl_file), format='turtle')
            parsed_count += 1
        except Exception as e:
            failed_count += 1
            print(f'  FAILED: {ttl_file.name} — {str(e)[:100]}')
    
    per_jur_graphs[jur] = g
    per_jur_stats[jur] = {
        'files_parsed': parsed_count,
        'files_failed': failed_count,
        'triples': len(g),
    }
    
    print(f'  Files parsed: {parsed_count}/{len(jur_files)}')
    print(f'  Total triples: {len(g)}')

In [ ]:
for jur in JURISDICTIONS:
    output_path = OUTPUT_DIR / f'merged_{jur}.ttl'
    per_jur_graphs[jur].serialize(destination=str(output_path), format='turtle')
    print(f'Saved: {output_path.name} ({len(per_jur_graphs[jur])} triples)')

# Summary
print('\n' + '=' * 60)
print('PER-JURISDICTION MERGE SUMMARY')
print('=' * 60)
total_triples = 0
for jur in JURISDICTIONS:
    stats = per_jur_stats[jur]
    total_triples += stats['triples']
    print(f'  {jur}: {stats["files_parsed"]} files, {stats["triples"]} triples')
print(f'\nTotal: {total_triples} triples across 4 jurisdictions')